# Assignment 5: NDVI Analysis Tracker

**Objective:** Analyze NDVI (Normalized Difference Vegetation Index) for agricultural fields using Landsat 8/9 imagery.

---

## Project Phases

| Phase | Task | Status |
|-------|------|--------|
| 1 | Download field boundaries | ✅ |
| 2 | Acquire rasters (Red, NIR) | ⏳ |
| 3 | Overlay Red band on fields | ⏳ |
| 4 | Overlay NIR band on fields | ⏳ |
| 5 | Calculate NDVI | ⏳ |
| 6 | Generate PNG visualizations | ⏳ |
| 7 | Zonal statistics | ⏳ |
| 8 | Analysis & summary | ⏳ |

## Environment Setup

In [ ]:
# Install dependencies (run once)
# !uv pip install landsatxplore rasterio geopandas numpy matplotlib shapely

## Phase 1: Download Field Boundaries

In [ ]:
from field_boundaries import download_fields, get_summary
import geopandas as gpd
from pathlib import Path

Path('data').mkdir(exist_ok=True)

# Download fields from corn belt
fields = download_fields(
    count=20,
    regions=['corn_belt'],
    crops=['corn', 'soybeans'],
    output_path='data/fields_cornbelt_2024.geojson'
)

# Summary
summary = get_summary(fields)
print(f"Total fields: {summary['total_fields']}")
print(f"Total area: {summary['total_area_acres']:.1f} acres")

## Phase 2: Acquire Rasters & Overlay (Red, NIR, NDVI)

### Prerequisites

**Option 1 - Real Landsat Data (requires USGS credentials):**
1. Get free USGS EarthExplorer account: https://ers.cr.usgs.gov/login
2. Add to `.env` file:
   ```
   USGS_USERNAME=your_username
   USGS_PASSWORD=your_password
   ```

**Option 2 - Demo Data (no credentials needed):**
- Script automatically generates synthetic rasters for demonstration

In [ ]:
# Run the overlay script (handles both real and demo data)
!python scripts/overlay_fields_raster.py

### Output Files

Generated in `output/` directory:
| File | Description |
|------|-------------|
| `FIELD_0001_red_band.png` | Red band (B4) with field overlay |
| `FIELD_0001_nir_band.png` | NIR band (B5) with field overlay |
| `FIELD_0001_ndvi.png` | NDVI choropleth with field overlay |

## Phase 3-7: Detailed Workflow (Reference)

*The following cells contain the detailed step-by-step workflow for reference.*

### Phase 3: Search Landsat 8/9 Scenes

In [ ]:
import os
from landsatxplore.api import API
import geopandas as gpd

# Load field boundaries
fields = gpd.read_file('data/fields_cornbelt_2024.geojson')
bbox = fields.total_bounds

# Connect to USGS API
api = API(os.environ['USGS_USERNAME'], os.environ['USGS_PASSWORD'])

# Search scenes (June-August 2024)
scenes = api.search(
    dataset='landsat_ot_c2_l2',
    bbox=(bbox[1], bbox[0], bbox[3], bbox[2]),
    start_date='2024-06-01',
    end_date='2024-08-31',
    max_cloud_cover=20
)

print(f"Found {len(scenes)} scenes")
for scene in scenes[:5]:
    print(f"  {scene['display_id']} — cloud: {scene['cloud_cover']}%")

api.logout()

## Phase 3: Download Imagery Bands

In [ ]:
from landsatxplore.earthexplorer import EarthExplorer
import os

# Best scene (lowest cloud cover)
best_scene = sorted(scenes, key=lambda s: s['cloud_cover'])[0]
print(f"Downloading: {best_scene['display_id']}")

ee = EarthExplorer(os.environ['USGS_USERNAME'], os.environ['USGS_PASSWORD'])

Path('data/landsat').mkdir(parents=True, exist_ok=True)
ee.download(best_scene['entity_id'], output_dir='data/landsat/')

ee.logout()
print("Download complete!")

## Phase 4: Clip Bands to Field Boundaries

In [ ]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import json
from pathlib import Path

fields = gpd.read_file('data/fields_cornbelt_2024.geojson')

# Find downloaded band files
band_files = list(Path('data/landsat').glob('*_SR_B4.TIF'))
print(f"Found {len(band_files)} B4 bands")

# Clip B4 (Red) and B5 (NIR) to each field
for idx, field in fields.iterrows():
    field_id = field['field_id']
    geom = [json.loads(gpd.GeoSeries([field.geometry]).to_json())['features'][0]['geometry']]
    
    for band in ['B4', 'B5']:
        band_path = f'data/landsat/{best_scene["display_id"]}_SR_{band}.TIF'
        out_path = Path(f'data/landsat/clipped_{field_id}_{band}_EPSG4326.tif')
        
        with rasterio.open(band_path) as src:
            out_image, out_transform = mask(src, geom, crop=True, nodata=0)
            out_meta = src.meta.copy()
        
        out_meta.update({
            'driver': 'GTiff',
            'height': out_image.shape[1],
            'width': out_image.shape[2],
            'transform': out_transform,
            'compress': 'lzw'
        })
        
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, 'w', **out_meta) as dst:
            dst.write(out_image)
        
        print(f"Clipped {band} for {field_id}")

## Phase 5: Calculate NDVI

In [ ]:
import rasterio
import numpy as np
from pathlib import Path

def calculate_ndvi(red_path: str, nir_path: str, output_path: str) -> str:
    """Calculate NDVI = (NIR - Red) / (NIR + Red)"""
    with rasterio.open(red_path) as red_src:
        red = red_src.read(1).astype('float32')
        profile = red_src.profile.copy()

    with rasterio.open(nir_path) as nir_src:
        nir = nir_src.read(1).astype('float32')

    denominator = nir + red
    ndvi = np.where(denominator > 0, (nir - red) / denominator, np.nan)

    profile.update(dtype='float32', count=1, compress='lzw', nodata=np.nan)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(ndvi, 1)

    print(f"NDVI saved: {output_path}")
    print(f"  min={np.nanmin(ndvi):.3f}  max={np.nanmax(ndvi):.3f}  mean={np.nanmean(ndvi):.3f}")
    return output_path

# Calculate NDVI for each field
fields = gpd.read_file('data/fields_cornbelt_2024.geojson')
for idx, field in fields.iterrows():
    field_id = field['field_id']
    red_path = f'data/landsat/clipped_{field_id}_B4_EPSG4326.tif'
    nir_path = f'data/landsat/clipped_{field_id}_B5_EPSG4326.tif'
    out_path = f'data/landsat/clipped_{field_id}_NDVI_EPSG4326.tif'
    
    if Path(red_path).exists() and Path(nir_path).exists():
        calculate_ndvi(red_path, nir_path, out_path)

## Phase 6: Zonal Statistics per Field

In [ ]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
import pandas as pd
import json

def zonal_stats(raster_path: str, fields_path: str) -> pd.DataFrame:
    """Extract per-field statistics from NDVI raster"""
    fields = gpd.read_file(fields_path)
    results = []

    with rasterio.open(raster_path) as src:
        for idx, field in fields.iterrows():
            geom = [json.loads(gpd.GeoSeries([field.geometry]).to_json())['features'][0]['geometry']]
            out_image, _ = mask(src, geom, crop=True, nodata=np.nan)
            data = out_image[0]
            valid = data[~np.isnan(data)]
            
            if len(valid) > 0:
                results.append({
                    'field_id': field.get('field_id', f'field_{idx}'),
                    'mean_ndvi': float(np.mean(valid)),
                    'std_ndvi': float(np.std(valid)),
                    'min_ndvi': float(np.min(valid)),
                    'max_ndvi': float(np.max(valid)),
                    'median_ndvi': float(np.median(valid)),
                    'pixel_count': len(valid),
                })

    return pd.DataFrame(results)

# Get stats for each field
ndvi_stats = []
fields = gpd.read_file('data/fields_cornbelt_2024.geojson')
for idx, field in fields.iterrows():
    field_id = field['field_id']
    ndvi_path = f'data/landsat/clipped_{field_id}_NDVI_EPSG4326.tif'
    
    if Path(ndvi_path).exists():
        with rasterio.open(ndvi_path) as src:
            data = src.read(1)
            valid = data[~np.isnan(data)]
            if len(valid) > 0:
                ndvi_stats.append({
                    'field_id': field_id,
                    'mean_ndvi': np.mean(valid),
                    'std_ndvi': np.std(valid),
                    'min_ndvi': np.min(valid),
                    'max_ndvi': np.max(valid),
                    'median_ndvi': np.median(valid),
                    'pixel_count': len(valid),
                })

stats_df = pd.DataFrame(ndvi_stats)
print(stats_df)
stats_df.to_csv('data/ndvi_statistics.csv', index=False)
print("\nSaved to data/ndvi_statistics.csv")

## Phase 7: Visualization & Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

# Load statistics
stats = pd.read_csv('data/ndvi_statistics.csv')
fields = gpd.read_file('data/fields_cornbelt_2024.geojson')

# Merge with field data
merged = fields.merge(stats, on='field_id')

# Plot 1: NDVI distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(merged['mean_ndvi'], bins=10, edgecolor='black')
axes[0].set_xlabel('Mean NDVI')
axes[0].set_ylabel('Field Count')
axes[0].set_title('Distribution of Mean NDVI by Field')

# Plot 2: Fields map colored by NDVI
merged.plot(column='mean_ndvi', cmap='RdYlGn', legend=True, ax=axes[1])
axes[1].set_title('Field NDVI Map')

plt.tight_layout()
plt.savefig('data/ndvi_analysis.png', dpi=150)
plt.show()

print("Visualization saved to data/ndvi_analysis.png")

## Summary

**Completed:**
- Field boundaries downloaded
- Landsat scenes searched
- Imagery bands clipped
- NDVI calculated
- Zonal statistics extracted
- Visualization generated

**Key Findings:**
- Average NDVI: ...
- Range: ...
- Crop correlation: ...